In [3]:
from pydantic import BaseModel, model_validator, ValidationError 
import pandas as pd 



In [24]:
class raw_data_cleanup(BaseModel):

    user_id: int 
    tenure_months: int
    monthly_charges: float
    total_support_tix: int
    is_churned: int 

    @model_validator(mode='before')
    @classmethod
    def data_check(cls, data: dict) -> dict:
        return_dict = {}

        user_id_actual = data.get("user_id") or data.get("account_id")
        if user_id_actual in ['NA', 'na', 'N/A', 'n/a', None, 'null', 'Null']:
            raise ValueError("Invalid user_id")
        if not isinstance(user_id_actual, int): 
            try:
                user_id_actual = int(user_id_actual)
            except:
                raise ValueError("Invalid user_id")
        return_dict['user_id'] = user_id_actual

        tenure_month_actual = data.get("tenure_months") or data.get("months_active") 
        if tenure_month_actual in ['NA', 'na', 'N/A', 'n/a', None, 'null', 'Null']:
            raise ValueError("Invalid tenure_month")
        if not isinstance(tenure_month_actual, int):
            try:
                tenure_month_actual = int(tenure_month_actual)
            except: 
                raise ValueError("Invalid tenure_month")
        return_dict['tenure_months'] = tenure_month_actual 

        monthly_charges_actual = data.get("monthly_charges")
        if monthly_charges_actual in ['NA', 'na', 'N/A', 'n/a', None, 'null', 'Null']:
            raise ValueError("Invalid monthly charge")       
        if not isinstance(monthly_charges_actual, float):
            try:
                monthly_charges_actual = float(monthly_charges_actual)
            except:
                raise ValueError("Invalid monthly charge")
        if monthly_charges_actual < 0:
            raise ValueError("Invalid monthly charge")
        return_dict['monthly_charges'] = monthly_charges_actual 

        support_tix_actual = data.get("total_support_tickets")
        if support_tix_actual in ['NA', 'na', 'N/A', 'n/a', None, 'null', 'Null']:
            raise ValueError("Invalid total_support_tickets")       
        if not isinstance(support_tix_actual, int):
            try: 
                support_tix_actual = int(support_tix_actual)
            except:
                raise ValueError("Invalid total_support_tickets")    
        return_dict['total_support_tix'] = support_tix_actual

        is_churned_actual = data.get("is_churned")
        if is_churned_actual in ['NA', 'na', 'N/A', 'n/a', None, 'null', 'Null']:
            raise ValueError("Invalid is_churn")
        if not isinstance(is_churned_actual, int):
            try:
                is_churned_actual = int(is_churned_actual)
            except:
                raise ValueError("Invalid is_churn")
        return_dict['is_churned'] = is_churned_actual 

        return return_dict 

In [27]:
valid_data = []
invalid_data = []

for index, data in enumerate(raw_churn_payload, start=1):
    try:
        record = raw_data_cleanup(**data)
        print(f"index: {index} success")
        valid_data.append(record)
    except ValidationError as e:
        print(e.errors()[0]['msg']) 
        

index: 1 success
index: 2 success
Value error, Invalid monthly charge
Value error, Invalid monthly charge


In [6]:
raw_churn_payload = [
    # Source A: Clean inputs, string IDs
    {
        "user_id": "1001",
        "tenure_months": 12,
        "monthly_charges": "65.50",
        "total_support_tickets": 2,
        "is_churned": 0
    },
    # Source B: Alternative keys ('account_id', 'months_active'), string floats
    {
        "account_id": "1002",
        "months_active": " 24 ",
        "monthly_charges": 89.99,
        "total_support_tickets": "0",
        "is_churned": "0"
    },
    # Source A: Negative monthly charges (Dirty)
    {
        "user_id": 1003,
        "tenure_months": 3,
        "monthly_charges": -15.00,
        "total_support_tickets": 5,
        "is_churned": 1
    },
    # Source B: Invalid numeric string for charges (Dirty)
    {
        "account_id": "1004",
        "months_active": 6,
        "monthly_charges": "N/A",
        "total_support_tickets": 1,
        "is_churned": 1
    }
]